# CodeBERT experiments in Google Colab
Run with a GPU runtime. This notebook trains the synthetic eight-class and CodeContests binary experiments and copies outputs to Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = 'https://github.com/burymewithmykatana/programming-error-pattern-recognition.git'
BRANCH = 'codex/presentation-ready'
!git clone --branch $BRANCH $REPO_URL /content/project
%cd /content/project
!pip install -q -e '.[transformer,datasets]'

Place the prepared split folders in `/content/drive/MyDrive/error-pattern-data/`. Use the reduced row limits below if runtime is constrained.

In [ ]:
from pathlib import Path
DATA = Path('/content/drive/MyDrive/error-pattern-data')
OUTPUT = Path('/content/drive/MyDrive/error-pattern-results')
OUTPUT.mkdir(parents=True, exist_ok=True)
assert (DATA / 'synthetic/train.csv').exists()
assert (DATA / 'synthetic/test.csv').exists()
assert (DATA / 'code_contests/train.csv').exists()
assert (DATA / 'code_contests/test.csv').exists()

In [ ]:
!python scripts/run_experiment.py \
  --train-data "$DATA/synthetic/train.csv" \
  --test-data "$DATA/synthetic/test.csv" \
  --model-type codebert --config configs/deep_model.yaml \
  --output "$OUTPUT/synthetic_codebert"

In [ ]:
!python scripts/run_experiment.py \
  --train-data "$DATA/code_contests/train.csv" \
  --test-data "$DATA/code_contests/test.csv" \
  --model-type codebert --config configs/deep_model.yaml \
  --output "$OUTPUT/codecontests_codebert"

## Reduced-data fallback
If Colab runs out of memory or time, sample each training class to 2,000 rows, keep the official test set unchanged, and rerun. Record the reduced row count in the report.

In [ ]:
import pandas as pd
def reduced_train(source, destination, rows_per_class=2000):
    frame = pd.read_csv(source)
    reduced = frame.groupby('label', group_keys=False).apply(
        lambda group: group.sample(min(len(group), rows_per_class), random_state=42),
        include_groups=False,
    ).reset_index(drop=True)
    reduced.to_csv(destination, index=False)
    return reduced['label'].value_counts()
# Example:
# reduced_train(DATA/'code_contests/train.csv', '/content/codecontests_reduced.csv')